# Lektion 09 - Metakognition Designmønster


## Opsætning

Denne notesbog demonstrerer designmønsteret Metacognition ved hjælp af Microsoft Agent Framework.

**Forudsætninger:**
- Azure OpenAI-implementering konfigureret via miljøvariabler
- Azure CLI autentificeret (`az login`)


In [ ]:
! pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

In [ ]:

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Hvad er Metakognition?

Metakognition er **tænkning om tænkning**. I forbindelse med AI-agenter betyder det at bygge agenter, der kan:

- **Selvreflektere** over deres egne resultater og resoneringsproces
- **Opdage fejl** og håndtere dem i stedet for at fejle uden at give besked
- **Vurdere** om deres svar er komplette og nyttige
- **Tilpasse** deres strategi, når en indledende tilgang ikke virker (f.eks. at falde tilbage på et backupsystem)

En metakognitiv agent svarer ikke kun på spørgsmål — den overvåger sin egen ydeevne og tilpasser sig løbende.


## Primære og backupværktøjer

Et almindeligt metakognitionsmønster er **fallback-strategien**. Agenten prøver først et primært værktøj; hvis det fejler (f.eks. en 404-fejl), genkender agenten fejlen og skifter gennemsigtigt til et backupværktøj.

Dette afspejler systemer i den virkelige verden, hvor primære tjenester kan være utilgængelige, og agenter må selv diagnosticere problemet, før de vælger en alternativ vej.

Nedenfor definerer vi to værktøjer til flyopslag:
- **Primær** — dækker Paris, Tokyo og Barcelona
- **Backup** — dækker Berlin, Sydney og New York City


In [ ]:
@tool(approval_mode="never_require")
def get_flight_times(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times for a destination (primary source)."""
    flights = {
        "Paris": "Departures: 08:00, 12:30, 17:45 — from $350",
        "Tokyo": "Departures: 11:00, 23:30 — from $890",
        "Barcelona": "Departures: 07:15, 14:00, 19:30 — from $280",
    }
    if destination in flights:
        return flights[destination]
    raise Exception(f"404: No flights found for {destination} in primary system")


@tool(approval_mode="never_require")
def get_flight_times_backup(
    destination: Annotated[str, "The destination city"]
) -> str:
    """Get available flight times from backup system (used when primary fails)."""
    backup_flights = {
        "Berlin": "Departures: 09:00, 16:00 — from $220",
        "Sydney": "Departures: 22:00 — from $1200",
        "New York City": "Departures: 06:00, 10:30, 15:00, 20:00 — from $450",
    }
    return backup_flights.get(
        destination,
        f"No flights found for {destination} in any system. Please try again later.",
    )

## Selvreflekterende agent med fejlgenopretning

Agenten nedenfor er instrueret til først at prøve det primære flyvesystem, genkende fejl og gennemsigtigt falde tilbage til backupsystemet. Efter hvert svar evaluerer den kort, om det fuldt ud besvarede brugerens spørgsmål.


In [ ]:
agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="FlightBookingAgent",
    instructions="""You are a flight booking agent with self-reflection capabilities.

When looking up flights:
1. Try the primary flight system first (get_flight_times)
2. If the primary system fails (404 error), acknowledge the error and try the backup system (get_flight_times_backup)
3. Always explain to the user what happened — be transparent about fallbacks
4. If both systems fail, apologize and suggest alternatives

After each response, briefly evaluate whether your answer was complete and helpful.""",
)

# Test with a destination in primary system
print("=== Test 1: Destination in primary system ===")
response = await agent.run(
    "What flights are available to Paris?",
    )
print(response)

# Test with a destination only in backup system
print("\n=== Test 2: Destination only in backup system ===")
response = await agent.run(
    "What flights are available to Berlin?",
    )
print(response)

## Selvevalueringsmønster

En anden facet af metakognition er **selvevaluering**: en separat agent (eller den samme agent i en anden gennemgang) gennemgår et svar for fuldstændighed, nøjagtighed og hjælpsomhed.

Nedenfor opretter vi en `ResponseEvaluator` agent, der scorer rejseagentens svar på tre dimensioner.


In [ ]:
evaluation_agent = await provider.create_agent(
    tools=[get_flight_times, get_flight_times_backup],
    name="ResponseEvaluator",
    instructions="""You are a quality evaluator for travel agent responses.
Given a travel question and the agent's response, evaluate:
1. Completeness: Did it answer all parts of the question? (1-5)
2. Accuracy: Is the information correct? (1-5)
3. Helpfulness: Would a traveler find this useful? (1-5)
Provide a brief evaluation with scores and one suggestion for improvement.""",
)

# Evaluate the agent's response from Test 1
eval_prompt = f"""Question: What flights are available to Paris?
Agent Response: {response}

Please evaluate the above response."""

evaluation = await evaluation_agent.run(eval_prompt)
print("=== Self-Evaluation ===")
print(evaluation)

## Sammendrag

I denne lektion lærte du, hvordan du bygger **metakognitive agenter** ved hjælp af Microsoft Agent Framework:

- **Selvrefleksion**: Agenter der overvåger deres egen ræsonnement og gennemsigtigt kommunikerer, hvad der skete.
- **Fejlgenopretning med fallback**: Et primært + backup-værktøjsmønster hvor agenten opdager fejl (f.eks. 404-fejl) og automatisk forsøger en alternativ kilde.
- **Selvevaluering**: En separat evalueringsagent, der vurderer svar ud fra fuldstændighed, nøjagtighed og hjælpsomhed.

Disse mønstre gør agenter mere robuste, gennemsigtige og troværdige — afgørende egenskaber for produktionsimplementeringer.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Ansvarsfraskrivelse**:
Dette dokument er oversat ved hjælp af AI-oversættelsestjenesten [Co-op Translator](https://github.com/Azure/co-op-translator). Selvom vi bestræber os på nøjagtighed, skal du være opmærksom på, at automatiske oversættelser kan indeholde fejl eller unøjagtigheder. Det originale dokument på dets oprindelige sprog bør betragtes som den autoritative kilde. For kritiske oplysninger anbefales professionel menneskelig oversættelse. Vi kan ikke holdes ansvarlige for misforståelser eller fejltolkninger, der måtte opstå som følge af brugen af denne oversættelse.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
